## Bias Evaluation in Machine Learning

In machine learning, bias evaluation is the process of detecting, measuring, and analyzing unfair or discriminatory behavior in artificial intelligence models.

Its main objective is to determine whether a model produces unfavorable or unequal outcomes for certain groups based on sensitive characteristics such as:
* gender,
* age,
* ethnicity,
* socioeconomic status.

Bias evaluation is therefore used to assess the fairness of Machine Learning system.

Unlike model training, bias evaluation does not directly learn from data. Instead, it analyzes whether the model's predictions are fair, balanced, and non-discriminatory.

## 1. Key Concepts

### 1.1 Algorithmic Bias

Algorithmic bias occurs when a model systematically produces unfair, imbalanced decisions or discriminatory outcomes toward specifics groups. These errors often reflect existing societal inequalities related to race, gender, age or socioeconomic status, and can originate at any stage of AI life cycle from data collection and model design to deployment (IBM,2026;EBSCO Research Staters; ScienceDirect, 2024).

Example: A recruitment system trained predominantly on male profiles may systematically favor male candidates over female candidates,  not because they are more qualified, but because the training data reflected historical hiring bias.

Reference:  [IBM AI Bias](https://www.ibm.com/think/topics/algorithmic-bias); [NIST AI Bias Resource Center](https://www.nist.gov/artificial_intelligence/ai-bias)

### 1.2 Fairness 

Fairness is the property expected of an equitable model, ensuring that its decisions do not unjustly disadvantage one group over another. Three formal definitions have gained prominence in the literature: **anti-classification** (protected attributes are not used to make decisions), **classification  parity** (predictive performance measures are equal across groups), and **calibration** (outcomes are independent of protected attributes given risk estimates).The objective is to avoid discrimination, statistical imbalance, and unjustified preferential treatment. However, as **Chouldechova (2017)** established, no single notion of fairness applies universally, and these definitions are mathematically incompatible with each other.

References: Barocas, Hardt & Narayanan, Fairness and Machine Learning, MIT Press (2023); Corbett-Davies & Goel (2018); Chouldechova (2017)

### 1.3 Sensitive Attributes

Sensitive attributes are characteristics of individuals on which a model must not discriminate. They can be explicit in the data (directly present) or implicit through proxy variables; for instance zip code has been shown to serve as a proxy for race, and credit rating as a proxy for safe driving (**Le Quy et al., 2022**). They may also be missing due to privacy restrictions, noisy due to flawed labeling, or encoded indirectly through correlated features.

Examples: gender, ethnicity, age, disability status, religion, national origin.

Bias evaluation compares model performance across these groups to detect unfair disparities.

References: Le Quy et al., WIREs Data Mining and Knowledge Discovery(2022); Transactions on Machine Learning Research(2024)

### 1.4 Fairness Metrics

Fairness metrics are quantitative measures used to evaluate the level of bias in a model. They are calculated from confusion matrix values: true positives (TP), false positives (FP), true negatives (TN) and false negatives (FN); across sensitive groups. They fall into two families:
* **Group fairness**: compares outcomes between demographic groups
* **Individual fairness**: requires that two similar individuals receive similar predictions (Dwork et al., 2012)

Common metrics include:
| Metric | Description |
|--------|-------------|
| **Demographic Parity** | The positive prediction rate must be equal across groups |
| **Equal Opportunity** | The true positive rate must be equal across groups |
| **Equalized Odds** | Both TPR and FPR must be equal across groups|
| **Disparate Impact** | Ratio of positive outcomes between groups - below 0.8 signals bias |

As proven by both impossibility theorems, no method can satisfy all fairness conditions simultaneously (Kleinberg et al., 2016; Chouldechova, 2017).

References: [Fairlearn Documentation](https://www.fairlearn.org); [Pagano et al.](https://www.mdpi.com/doi/10.1002/widm.1452), BDCC(2023); [Kleinberg et al.(2016)](https://arxiv.org/abs/1609.05807); [Dwork et al.(2012)](https://arxiv.org/abs/1703.00056)

## 2. Pseudo-algorithm

INPUT : Trained model (M), Test data (D), Sensitive attributes (A (gender, age, race), Fairness threshold (T)

OUTPUT : Bias report (metrics per subgroup, status BIASED / FAIR)

BEGIN

###### 1. INITIALIZATION

. Identify the sensitive attributes (A) to test
. Define the privileged group (G_priv)
. Define the unprivileged group (G_unpriv)

###### 2. PREDICTION

. Generate predictions P of model M on data D; P <-- M.predict(D)

###### 3. PERFORMANCE METRICS COMPUTATION PER SUBGROUP

For each group g in {G_priv, G_unpriv}:

. **TPR_g** (True positive Rate) = TP / (TP + FN)

. **FPR_g** (False positive Rate) = FP / (FP + TN)

. **PPR_g** (Positive Prediction Rate) = Nb "Yes" / Total Nb

###### 4. FAIRNESS EVALUATION (BIAS DETECTION)

. **Disparate Impact** = PPR_unpriv / PPR_priv  --> Bias if ratio < 0.8

. **Equal Opportunity** = |TPR_priv - TPR_unpriv| --> Bias if difference > T

. **Equalized Odds** = (|TPR_priv - TPR_unpriv|, |FDR_priv - FDR_unpriv|) --> Bias if either difference > T

. **Demographic Parity** = PPR_priv - PPR_unpriv --> Bias if difference > T

###### 5. COUNTERFACTUAL ANALYSIS (OPTIONAL)

For a given input x:

    . Flip the sensitive attribute A (eg: change 'male' --> 'female')
    
    . Compute P_original <-- M.predict(x)
    
    . Compute P_modified <-- M.predict(x')
    
    . If P_original != P_modified --> bias detected
    
###### 6. REPORT GENERATION

For each computed metric m:

    IF |m| > T:
    
        --> Flag as BIASED in the report 
        
    ELSE:
    
        --> Flag as FAIR in the report 
        
    Report <-- {metric, value, status, subgroup }
    
    RETURN Report
    
END
    
    

## 3. Fairness Implementation

    For this section, we use the Adult Income Dataset (UCI machine Learning repository), one of the most widely used datasets in fairness research. It contains 48,842 samples describing individuals based on census data, with features such as age, education, occupation, marital status, race and gender. The prediction task is to determine whether a person earns more or less than 50,000 USD per year. Because it contains real sensitive attributes, particularly sex and race, it is the reference dataset used in fairness evaluation 
[Barocas, Hardt & Narayanan](https://fairmlbook.org); [Fairlean Documentation](https://www. fairlearn.org)

###### Dataset loading and preparation

In [1]:
import os
import time
import numpy as np
import pandas as pd

from ifri_mini_ml_lib.metrics.fairness import (
    selection_rate,
    selection_rate_per_group,
    tpr_fpr_by_group,
    demographic_parity_ratio,
    demographic_parity_difference,
    equalized_odds_difference,
    equalized_odds_ratio,
)
from ifri_mini_ml_lib.preprocessing.preparation import CategoricalEncoder, DataSplitter
from ifri_mini_ml_lib.classification import DecisionTree

# ── Fairlearn for comparison ─────────────────
from fairlearn.metrics import (
    demographic_parity_ratio as sk_dpr,
    demographic_parity_difference as sk_dpd,
    equalized_odds_difference as sk_eod,
    equalized_odds_ratio as sk_eor,
    MetricFrame,
    true_positive_rate,
    false_positive_rate,
)

# Load Adult Income Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship',
    'race', 'sex', 'capital_gain', 'capital_loss',
    'hours_per_week', 'native_country', 'income'
]

df = pd.read_csv(url, names=columns, sep=',\s*', engine='python')

# Encode target
df['income'] = (df['income'] == '>50K').astype(int)

# Encode categorical columns with CategoricalEncoder
categorical_cols = [
    'workclass', 'education', 'marital_status',
    'occupation', 'relationship', 'race',
    'sex', 'native_country'
]

encoder = CategoricalEncoder(encoding_type='label')
df[categorical_cols] = encoder.fit_transform(df[categorical_cols])

# Features, target, sensitive attribute — keep as DataFrame/Series for DataSplitter
X = df.drop('income', axis=1)
y = df['income']
groups = df['sex']

# Train/test split with ifri_mini_ml_lib
splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.3)
_, g_test = splitter.train_test_split(groups, test_size=0.3)

# Convert to numpy arrays for DecisionTree
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values
g_test = g_test.values

# Train biased model — trained only on male data
male_col_index = list(df.drop('income', axis=1).columns).index('sex')
male_mask = X_train[:, male_col_index] == 1

model = DecisionTree()
model.fit(X_train[male_mask], y_train[male_mask])
y_pred = model.predict(X_test)

print("Dataset loaded successfully !")
print(f"Test set size : {len(y_test)} samples")
print(f"Groups : 0 = Female, 1 = Male")
print(f"Positive rate : {y_test.mean():.3f}")

Dataset loaded successfully !
Test set size : 9768 samples
Groups : 0 = Female, 1 = Male
Positive rate : 0.237


**Note:** Since the DecisionTree model trained on this dataset does not produce sufficiently contraste predictions between groups to clearly illustrate bias, we deliberatly introduce a synthetic bias in th predictions. Males (group = 1) receive positives predictions 70% of the time, while females (group = 0) receive them only 20% of the time. This allows us to clearly demonstrate how the fairness metrics detect and quantify discrimination between groups.    

In [2]:
np.random.seed(42)
y_pred = np.where(g_test == 1,
                  np.random.choice([0, 1], size=len(g_test), p=[0.3, 0.7]),
                  np.random.choice([0, 1], size=len(g_test), p=[0.8, 0.2]))

In [3]:
print("Groups in test set :", np.unique(g_test, return_counts=True))
print("y_pred distribution :", np.unique(y_pred, return_counts=True))

Groups in test set : (array([0, 1]), array([6502, 3266]))
y_pred distribution : (array([0, 1]), array([6133, 3635]))


###### 3.1 selection_rate()

This function computes the proportion of positive predictions in the dataset. 

In [4]:
# ifri-mini-ml-lib
ifri_result = selection_rate(y_pred, pos_label=1)
print("ifri-mini-ml-lib :", ifri_result)

# Fairlearn equivalent
sk_result = np.mean(y_pred == 1)
print("Fairlearn equivalent :", sk_result)

ifri-mini-ml-lib : 0.37213349713349714
Fairlearn equivalent : 0.37213349713349714


**Interpretation:** About 37% of individuals are predicted to earn more than $50k/year.Both librairies return identical results.

###### 3.2 selection_rate_per_group()

This function computes the selection rate separately for each group;here, male(1) and female(0) 

In [5]:
# ifri-mini-ml-lib
ifri_result = selection_rate_per_group(y_pred, g_test, pos_label=1)
print("ifri-mini-ml-lib :", ifri_result)

# Fairlearn equivalent
sk_result = MetricFrame(
    metrics=lambda y_true, y_pred: np.mean(y_pred == 1),
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=g_test
).by_group
print("Fairlearn equivalent :", sk_result)

ifri-mini-ml-lib : {np.int64(0): np.float64(0.20455244540141496), np.int64(1): np.float64(0.7057562767911819)}


Fairlearn equivalent : sensitive_feature_0
0    0.204552
1    0.705756
Name: <lambda>, dtype: float64


**Interpretation:** Male receive positive predictions 3.4x more often than females (70.6% vs 20.5%), suggesting a significant gap between groups.

###### 3.3 demographic_parity_ratio()

This function measures whether all groups receive positive predictions at a similar rate. Computes $\frac{\min(PPR)}{\max(PPR)}$.

In [6]:
# ifri-mini-ml-lib
ifri_ratio, ifri_rates = demographic_parity_ratio(y_pred, g_test, pos_label=1)
print("ifri-mini-ml-lib ratio :", ifri_ratio)
print("ifri-mini-ml-lib rates :", ifri_rates)

# Fairlearn
sk_ratio = sk_dpr(y_test, y_pred, sensitive_features=g_test)
print("Fairlearn ratio :", sk_ratio)

ifri-mini-ml-lib ratio : 0.28983439769241703
ifri-mini-ml-lib rates : {np.int64(0): np.float64(0.20455244540141496), np.int64(1): np.float64(0.7057562767911819)}


Fairlearn ratio : 0.28983439769241703


**Interpretation:** Ratio = 0.2045/0.7057 = 0.2898; far below 0.8: **strong bias detected**. Males are 3.4x more likely to receive a positive prediction than females.

| **Value** | **Meaning**|
|-----------|--------------|
| =1.0 | Perfect fairness|
| >=0.8 | Acceptable |
| <0.8 | Bias detected |

###### 3.4 demographic_parity_difference()

    This function measures the absolute gap between groups. It computes |max(PPR) - min(PPR)|.

In [7]:
# ifri-mini-ml-lib
ifri_diff, ifri_rates = demographic_parity_difference(y_pred, g_test, pos_label=1)
print("ifri-mini-ml-lib diff :", ifri_diff)
print("ifri-mini-ml-lib rates :", ifri_rates)

# Fairlearn
sk_diff = sk_dpd(y_test, y_pred, sensitive_features=g_test)
print("Fairlearn diff :", sk_diff)

ifri-mini-ml-lib diff : 0.501203831389767
ifri-mini-ml-lib rates : {np.int64(0): np.float64(0.20455244540141496), np.int64(1): np.float64(0.7057562767911819)}
Fairlearn diff : 0.501203831389767


**Interpretation:** Gap = |0.7057 - 0.2045| = 0.5012; above 0.1:**bias detected**. There is 50% gap between the selection rates of males and females.

| **Value** | **Meaning** |
|-----------|-------------|
| =0.0 | Perfect fairness | 
| <=0.1 | Acceptable |
| >0.1 | Bias detected |

###### 3.5 tpr_fpr_by_group

This function computes TPR and FPR for each group. Used intrnally by equalisez odds functions.

TPR = $\frac{TP}{TP + FN}$     ;      FPR = $\frac{FP}{FP + TN}$

In [8]:
# ifri-mini-ml-lib
ifri_tpr, ifri_fpr = tpr_fpr_by_group(y_test, y_pred, g_test, pos_label=1)
print("ifri-mini-ml-lib TPR :", ifri_tpr)
print("ifri-mini-ml-lib FPR :", ifri_fpr)

# Fairlearn
sk_tpr = MetricFrame(
    metrics=true_positive_rate,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=g_test
).by_group
sk_fpr = MetricFrame(
    metrics=false_positive_rate,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=g_test
).by_group
print("Fairlearn TPR :", sk_tpr)
print("Fairlearn FPR :", sk_fpr)

ifri-mini-ml-lib TPR : {np.int64(0): np.float64(0.22443729903536977), np.int64(1): np.float64(0.7097625329815304)}
ifri-mini-ml-lib FPR : {np.int64(0): np.float64(0.19830200121285627), np.int64(1): np.float64(0.7045454545454546)}
Fairlearn TPR : sensitive_feature_0
0    0.224437
1    0.709763
Name: true_positive_rate, dtype: float64
Fairlearn FPR : sensitive_feature_0
0    0.198302
1    0.704545
Name: false_positive_rate, dtype: float64


**Interpretation:** The model correctly identifies 71% of high-income males(TPR = 0.710) but only 22% of high-income females(TPR = 0.224). FPR is also higher for males (70.5% vs 19.8%), meaning the model makes more false positive errors for males.

###### 3.6 equalized_odds_difference()

This function computes the maximum gap of TPR and FPR between groups.

EOD = max(|$TPR_{\max} - TPR_{\min}$|, |$FPR_{\max} - FPR_{\min}$|).

In [9]:
# ifri-mini-ml-lib
ifri_diff, ifri_tpr, ifri_fpr = equalized_odds_difference(y_test, y_pred, g_test, pos_label=1)
print("ifri-mini-ml-lib diff :", ifri_diff)
print("ifri-mini-ml-lib TPR :", ifri_tpr)
print("ifri-mini-ml-lib FPR :", ifri_fpr)

# Fairlearn
sk_diff = sk_eod(y_test, y_pred, sensitive_features=g_test)
print("Fairlearn diff :", sk_diff)

ifri-mini-ml-lib diff : 0.5062434533325983
ifri-mini-ml-lib TPR : {np.int64(0): np.float64(0.22443729903536977), np.int64(1): np.float64(0.7097625329815304)}
ifri-mini-ml-lib FPR : {np.int64(0): np.float64(0.19830200121285627), np.int64(1): np.float64(0.7045454545454546)}
Fairlearn diff : 0.5062434533325983


**Interpretation:** TPR gap =|0.7097 - 0.2244| =0.4853 ; FPR gap = |0.7045 - 0.1983|= 0.0.5062. Worst case = 0.5062; above 0.1: **Bias detected**.

| **Value** | **Meaning** |
|-----------|------------------|
| =0.0 | Perfect fairness |
| <=0.1 | Acceptable |
| >0.1 | Bias detected |

###### 3.7 equalized_odds_ratio()

This function computes min/max ratios of TPR and FPR between groups, returns the worst case.

EOR = min($\frac{\min(TPR)}{\max(TPR)}$, $\frac{\min(FPR)}{\max(FPR)}$)

In [10]:
# ifri-mini-ml-lib
ifri_ratio, ifri_tpr, ifri_fpr = equalized_odds_ratio(y_test, y_pred, g_test, pos_label=1)
print("ifri-mini-ml-lib ratio :", ifri_ratio)
print("ifri-mini-ml-lib TPR :", ifri_tpr)
print("ifri-mini-ml-lib FPR :", ifri_fpr)

# Fairlearn
sk_ratio = sk_eor(y_test, y_pred, sensitive_features=g_test)
print("Fairlearn ratio :", sk_ratio)

ifri-mini-ml-lib ratio : 0.2814609049472798
ifri-mini-ml-lib TPR : {np.int64(0): np.float64(0.22443729903536977), np.int64(1): np.float64(0.7097625329815304)}
ifri-mini-ml-lib FPR : {np.int64(0): np.float64(0.19830200121285627), np.int64(1): np.float64(0.7045454545454546)}
Fairlearn ratio : 0.2814609049472798


**Interpretation:** TPR ratio = 0.2244/0.7097 = 0.3162 ; FPR ratio = 0.1983/0.07045 = 0.2815. Worst case = 0.2815; far below 0.8 : **Bias detected**.

| **Value** | **meaning** |
|--------------|-------------|
| =1.0 | Perfect fairness |
| >=0.8 | Acceptable |
| <0.8 | Bias detected |

###### 3.8 Global performance and execution time comparison

In [11]:
# ifri-mini-ml-lib
start = time.time()
demographic_parity_ratio(y_pred, g_test, pos_label=1)
demographic_parity_difference(y_pred, g_test, pos_label=1)
equalized_odds_difference(y_test, y_pred, g_test, pos_label=1)
equalized_odds_ratio(y_test, y_pred, g_test, pos_label=1)
ifri_time = time.time() - start

# Fairlearn
start = time.time()
sk_dpr(y_test, y_pred, sensitive_features=g_test)
sk_dpd(y_test, y_pred, sensitive_features=g_test)
sk_eod(y_test, y_pred, sensitive_features=g_test)
sk_eor(y_test, y_pred, sensitive_features=g_test)
sk_time = time.time() - start

# Summary table
print(f"{'Metric':<30} | {'ifri-mini-ml-lib':>18} | {'Fairlearn':>10}")
print(f"{'-'*30}-|-{'-'*18}-|-{'-'*10}")
print(f"{'Demographic Parity Ratio':<30} | {'0.2898':>18} | {'0.2898':>10}")
print(f"{'Demographic Parity Diff':<30} | {'0.5012':>18} | {'0.5012':>10}")
print(f"{'Equalized Odds Difference':<30} | {'0.5062':>18} | {'0.5062':>10}")
print(f"{'Equalized Odds Ratio':<30} | {'0.2815':>18} | {'0.0.2815':>10}")
print(f"{'Execution Time (s)':<30} | {ifri_time:>18.6f} | {sk_time:>10.6f}")

Metric                         |   ifri-mini-ml-lib |  Fairlearn
-------------------------------|--------------------|-----------
Demographic Parity Ratio       |             0.2898 |     0.2898
Demographic Parity Diff        |             0.5012 |     0.5012
Equalized Odds Difference      |             0.5062 |     0.5062
Equalized Odds Ratio           |             0.2815 |   0.0.2815
Execution Time (s)             |           0.007618 |   0.926296


**Interpretation:**

**Correctness:** All metrics produce identical results between ifri-mini-ml-lib and Fairlearn, validating the correctness of the implementation.

**Performance:** ifri-mini-ml-lib is approximately **207x faster** than Fairlearn (0.005645s vs 1.171818s).

## 4.Real-life Applications

Because fairness metrics are essential in any responsible AI system, they are widely used across many domains.

1. **Criminal Justice:** Fairness metrics were used to analyze COMPAS, a recidivism prediction tool used in the US justice system, which was found to be biased against Black defendants (Chouldechova, 2017).

2. **Recruitment and HR:** Bias evaluation helps detect discrimination in automated hiring systems that may systematically disadvantage women or minority groups based on historical training data, exactly the type of bias highlighted by Adult Income Dataset used in this module.

3. **Healthcare:** Fairness metrics ensure that medical diagnosis models perform equally well across different demographic groups: age, gender or ethnicity; avoiding unequal treatment. (Barocas, Hardt & Narayanan, 2023)

4. **Finance and Credit Scoring:** Banks and financial institutions use fairness metrics to ensure that credit scoring models do not discriminate against applicants based on race, gender or origin.  

## 5. Limitations and Challenges

While fairness metrics are powerful tools, they come with important limitations that must be considered.

1. **Impossibility of simultaneous fairness:** As proven by Chouldechova (2017) and Kleinberg et al. (2016), it is mathematically impossible to satisfy all fairness criteria at the same time when base rates differ between groups. Every choice of metric reflects a value judgment about which type of error is more acceptable.

2. **Definition of privileged and unprivileged groups:**Defining which group is privileged requires domain knowledge and context. A wrong definition can lead to misleading conclusions.

3. **Proxy attributes:** Even when sensitive attributes are removed from the data, the model can still learn to discriminate through correlated proxy variables, making bias harder to detect (Le Quy et al., 2022).

4. **Binary group assumption:** Most metrics assume binary groups (eg: male/female), which does not reflect the complexity of real-world demographics with multiple overlapping groups.

## 6. References

.  [Barocas, S., Hardt, M., & Narayanan, A. (2023). *Fairness and Machine Learning*. MIT Press.](https://fairmlbook.org)

. [Chouldechova, A. (2017). Fair Prediction with Disparate Impact. *Big Data, 5(2), 153–163.*](https://arxiv.org/abs/1703.00056)

. [Kleinberg, J., Mullainathan, S., & Raghavan, M. (2016). Inherent Trade-Offs in the Fair Determination of Risk Scores.](https://arxiv.org/abs/1609.05807)

. [Fairlearn Documentation](https://www.fairlearn.org)

. [Dwork et al.(2012)](https://arxiv.org/abs/1104.3913)

. [Le Quy et al.(2022)](https://wires.onlinelibrary.wiley.com/doi/10.1002/widm.1452)

. [Pagano et al. (2023)](https://www.mdpi.com/2504-2289/7/1/14)

. [Adult Income Dataset (UCI)](https://archive.ics.uci.edu/ml/datasets/adult)

. [Corbett-Davies & Goel (2018)](https://arxiv.org/abs/1808.00023)